# Combine generators into a pipeline

The shape: each stage is a generator that takes an iterable and yields a transformed iterable. You wire stages together by passing each one's output into the next. Nothing runs until a *consumer* at the end (a `for`, `sum`, `list`, file write) starts pulling values through.

This is the functional-pipeline style for data processing. It's how you write things like UNIX shell pipelines, but in Python.

## The four basic stage shapes

Most pipelines are built from four kinds of stage:

1. **Source** — produces values. A generator with no input parameter (or one that opens a file). Examples: `read_lines(path)`, `query(db)`, `range(...)`.
2. **Filter** — yields a subset of its input. `(x for x in xs if predicate(x))`.
3. **Transform** — yields one value out per value in. `(f(x) for x in xs)`.
4. **Aggregate** — yields *fewer* values, often one (`sum`), or yields windowed/grouped chunks (`window`, `groupby`).

Most pipelines look like: source → filter → transform → ... → consumer. The consumer is whatever drives the chain — `for`, `sum`, write-to-file.

## A small worked example — sensor readings

Pretend we have a stream of sensor readings, each `(timestamp, sensor_id, value)`. We want the average reading per sensor for sensors whose values stayed above a noise threshold.

In [ ]:
import random

# Source: a generator that produces readings. In real code this might be
# reading from a file or a network socket.
def read_readings(n, seed=0):
    rnd = random.Random(seed)
    sensors = ['s1', 's2', 's3']
    for i in range(n):
        yield (i, rnd.choice(sensors), rnd.gauss(10, 2))


# Peek
from itertools import islice
print(list(islice(read_readings(20), 5)))

### Stage 1 — filter out the noise

In [ ]:
def above_threshold(readings, threshold):
    for r in readings:
        if r[2] > threshold:
            yield r

### Stage 2 — round timestamps into buckets

In [ ]:
def bucketise(readings, bucket_size):
    for ts, sensor, value in readings:
        bucket = (ts // bucket_size) * bucket_size
        yield (bucket, sensor, value)

### Stage 3 — group by (bucket, sensor) and average

Aggregation needs a sort + groupby step (because `groupby` only catches adjacent runs).

In [ ]:
from itertools import groupby
from statistics import mean

def average_per_bucket(readings):
    # sort by (bucket, sensor) so groups are contiguous
    keyed = sorted(((r[0], r[1]), r[2]) for r in readings)
    for key, group in groupby(keyed, key=lambda p: p[0]):
        values = [v for _, v in group]
        yield (*key, mean(values))

### Wire it up

In [ ]:
pipeline = average_per_bucket(
    bucketise(
        above_threshold(
            read_readings(200, seed=1),
            threshold=8.0,
        ),
        bucket_size=50,
    )
)

# The consumer: print the first ten results
for row in islice(pipeline, 10):
    print(row)

Each stage is small, named, and individually testable. To debug, drop a `for x in stage: print(x)` between any two stages.

## Composition style — one big function vs. many tiny ones

You *could* write the whole pipeline as one big generator:

```python
def big_thing():
    rnd = random.Random(0)
    for i in range(200):
        s = rnd.choice(['s1', 's2', 's3'])
        v = rnd.gauss(10, 2)
        if v > 8.0:
            ...
```

…but you lose the ability to swap, test, or reuse pieces. The small-functions version reads better and adapts when requirements change ("now also filter by sensor whitelist" — add one stage).

## Generator expressions vs functions

A one-line transform doesn't need a function — a generator expression is fine:

```python
above_threshold = (r for r in readings if r[2] > 8.0)
```

Two trade-offs:

- **Genexp**: anonymous, captures variables from the enclosing scope. Good for in-place tweaks.
- **Function**: parameterised, importable, testable. Good for stages you'll reuse.

Real pipelines mix the two: small genexps inline, named functions for the meaningful steps.

## Plumbing: writing a pipeline as a list of stages

When the pipeline is dynamic (which stages depend on user config), you can't hard-code the call chain. A `compose` helper that applies a list of stages in order is concise.

In [ ]:
from functools import reduce

def compose(source, *stages):
    '''Apply each stage to the running iterable in order.'''
    return reduce(lambda it, stage: stage(it), stages, source)


# Define stages as one-arg callables
def square(xs):
    for x in xs:
        yield x * x

def positive(xs):
    for x in xs:
        if x > 0:
            yield x


result = compose(range(-3, 4), positive, square)
print(list(result))    # [1, 4, 9]

For stages that need extra arguments, use `functools.partial`:

In [ ]:
from functools import partial, reduce

def above(xs, threshold):
    for x in xs:
        if x > threshold:
            yield x

stages = [partial(above, threshold=2), square]
print(list(compose(range(6), *stages)))   # [9, 16, 25]

## A debugging pattern — tap a stream

When values flow through a long pipeline and you want to see what's happening at one spot, drop in a `tap` stage that yields each value through but also prints (or logs) it. Doesn't break the pipeline.

In [ ]:
def tap(xs, label):
    for x in xs:
        print(f'[{label}] {x}')
        yield x


# Insert anywhere in the chain to inspect:
result = compose(
    range(-3, 4),
    positive,
    partial(tap, label='after positive'),
    square,
)
print(list(result))

## The consumer matters

A pipeline does no work until something asks for values. Make sure your consumer actually drives the chain:

- `list(pipeline)` — pulls everything, materialises.
- `for x in pipeline:` — pulls one at a time.
- `sum(pipeline)`, `max(pipeline)`, etc — pull and reduce.
- `next(pipeline)` — one value, then leave.

A common mistake is building a pipeline and then forgetting to consume it — the function returns, your `print` statements never fire, and you wonder why nothing happened.

In [ ]:
def shouts(xs):
    for x in xs:
        print(f'shouting {x}')
        yield x.upper()

# This does nothing — never consumed:
shouts(['hi', 'there'])
print('end of cell — no shouts printed')

In [ ]:
# This consumes — the prints fire:
list(shouts(['hi', 'there']))

## Summary

- Build pipelines from small stages: source, filter, transform, aggregate.
- Each stage is a generator function or a generator expression.
- Compose by nesting calls or by `reduce`-ing over a list of stages.
- Constant memory regardless of stream size; debug by tapping mid-stream.
- Nothing runs until a consumer pulls — always check that something is actually iterating the pipeline.